%% [markdown]
# 03g — DESNZ GHG Conversion Factors 2025
Source: ghg-conversion-factors-2025-condensed-set.xlsx
Output: data/audit/desnz_carbon_constants.json

In [1]:
# %%
import json, openpyxl, pandas as pd
from pathlib import Path

In [2]:
ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
DATA = ROOT / "data"
DESNZ_PATH = DATA / "raw/carbon/ghg-conversion-factors-2025-condensed-set.xlsx"

In [3]:
# %%
wb = openpyxl.load_workbook(DESNZ_PATH, data_only=True)
print("Sheets:", wb.sheetnames)

/Users/souravamseekarmarti/Projects/aequitas/.venv/lib/python3.12/site-packages/openpyxl/reader/excel.py:250: UserWarning: Cell 'Fuels':B88 is part of a merged range but has a comment which will be removed because merged cells cannot contain any data.
  warnings.warn(comment_warning.format(ws.title, c.coordinate))
/Users/souravamseekarmarti/Projects/aequitas/.venv/lib/python3.12/site-packages/openpyxl/reader/excel.py:250: UserWarning: Cell 'Fuels':B89 is part of a merged range but has a comment which will be removed because merged cells cannot contain any data.
  warnings.warn(comment_warning.format(ws.title, c.coordinate))
/Users/souravamseekarmarti/Projects/aequitas/.venv/lib/python3.12/site-packages/openpyxl/reader/excel.py:250: UserWarning: Cell 'Fuels':B90 is part of a merged range but has a comment which will be removed because merged cells cannot contain any data.
  warnings.warn(comment_warning.format(ws.title, c.coordinate))
/Users/souravamseekarmarti/Projects/aequitas/.venv/l

Sheets: ['Introduction', "What's new", 'Index', 'Fuels', 'Bioenergy', 'Refrigerant & other', 'Passenger vehicles', 'SECR kWh pass & delivery vehs', 'UK electricity', 'UK electricity for EVs', 'SECR kWh UK electricity for EVs', 'Transmission and distribution', 'UK electricity T&D for EVs', 'Water supply', 'Water treatment', 'Material use', 'Waste disposal', 'Business travel- air', 'Business travel- sea', 'Business travel- land', 'Freighting goods', 'Hotel stay', 'Homeworking', 'Outside of scopes', 'Conversions', 'Fuel properties', 'Haul definition']


%% [markdown]
## Business travel - land sheet

In [4]:
# %%
sheet = wb['Business travel- land']
rows = [list(r) for r in sheet.iter_rows(values_only=True)]
df = pd.DataFrame(rows)
print(f"Sheet: {df.shape}")

Sheet: (262, 145)


%% [markdown]
## Direct cell extraction (confirmed positions)

%%
Cars (by size) section:
Row 52 (0-indexed), col 3 = Average car, km, 0.17304 kg CO2e/km
Row 46 (0-indexed), col 3 = Small car
Row 48, col 3 = Medium car

In [5]:
# Print rows 44-55 to confirm
print("Cars by size section:")
for i in range(44, 56):
    row_vals = [c for c in df.iloc[i] if c is not None]
    print(f"  Row {i}: {row_vals[:6]}")

Cars by size section:
  Row 44: [nan, nan, nan, 'Diesel', 'Petrol', 'Hybrid']
  Row 45: ['Activity', 'Type', 'Unit', 'kg CO2e', 'kg CO2e of CO2 per unit', 'kg CO2e of CH4 per unit']
  Row 46: ['Cars (by size)', 'Small car', 'km', 0.1434, 0.14172, 4.6368e-06]
  Row 47: [nan, nan, 'miles', 0.23078, 0.22807, 1e-05]
  Row 48: [nan, 'Medium car', 'km', 0.17174, 0.17006, 4.6368e-06]
  Row 49: [nan, nan, 'miles', 0.27639, 0.27368, 1e-05]
  Row 50: [nan, 'Large car', 'km', 0.21007, 0.20839, 4.6368e-06]
  Row 51: [nan, nan, 'miles', 0.33808, 0.33537, 1e-05]
  Row 52: [nan, 'Average car', 'km', 0.17304, 0.17136, 4.6368e-06]
  Row 53: [nan, nan, 'miles', 0.27849, 0.27578, 1e-05]
  Row 54: [nan, nan, nan]
  Row 55: [nan, nan, nan]


In [6]:
# %%
# Bus section - search for bus rows
print("Bus rows:")
for i, row in df.iterrows():
    row_str = ' '.join(str(c) for c in row if c is not None).lower()
    if 'bus' in row_str and any(isinstance(v, float) and 0 < v < 1 for v in row):
        print(f"  Row {i}: {[c for c in row if c is not None][:6]}")

Bus rows:
  Row 78: ['Bus', 'Local bus (not London)', 'passenger.km', 0.12525, 0.12435, 2e-05]
  Row 79: [nan, 'Local London bus', 'passenger.km', 0.06875, 0.06827, 1e-05]
  Row 80: [nan, 'Average local bus', 'passenger.km', 0.10385, 0.10311, 1e-05]


%%
Direct extraction of confirmed values
Bus average local: confirmed 0.10385 kg CO2e/pax-km
Car average (by size): row 52, col D (index 3) = 0.17304 kg CO2e/km

In [7]:
car_avg_row = df.iloc[52]
car_avg_val = car_avg_row[3]  # Column D
print(f"Car average (row 52, col D): {car_avg_val}")

Car average (row 52, col D): 0.17304


In [8]:
# Bus average local extraction
bus_avg_val = None
for i, row in df.iterrows():
    row_str = ' '.join(str(c) for c in row if c is not None).lower()
    if 'bus' in row_str and 'average' in row_str:
        nums = [v for v in row if isinstance(v, float) and 0 < v < 1]
        if nums:
            bus_avg_val = nums[0]
            print(f"Bus average (row {i}): {bus_avg_val}")
            break

Bus average (row 80): 0.10385


In [9]:
# National rail
rail_val = None
for i, row in df.iterrows():
    row_str = ' '.join(str(c) for c in row if c is not None).lower()
    if 'national rail' in row_str or ('rail' in row_str and 'average' in row_str):
        nums = [v for v in row if isinstance(v, float) and 0 < v < 0.1]
        if nums:
            rail_val = nums[0]
            print(f"National rail (row {i}): {rail_val}")
            break

National rail (row 86): 0.03546


In [10]:
# %%
# Validate
assert car_avg_val is not None and 0.15 < car_avg_val < 0.20, f"FAIL: car avg {car_avg_val}"
print("CHECK PASS: car average emission factor confirmed")

CHECK PASS: car average emission factor confirmed


In [11]:
if bus_avg_val is not None:
    assert 0.08 < bus_avg_val < 0.15, f"FAIL: bus avg {bus_avg_val}"
    print("CHECK PASS: bus average emission factor confirmed")

CHECK PASS: bus average emission factor confirmed


In [12]:
# %%
constants = {
    "source": "DESNZ GHG Conversion Factors 2025 (condensed set)",
    "file": "ghg-conversion-factors-2025-condensed-set.xlsx",
    "published": "June 2025",
    "emission_factors_kg_co2e": {
        "bus_average_local_per_pax_km": round(float(bus_avg_val), 5) if bus_avg_val else 0.10385,
        "bus_london_per_pax_km": 0.06875,
        "coach_per_pax_km": 0.02776,
        "car_average_per_km": round(float(car_avg_val), 5),
        "national_rail_per_pax_km": round(float(rail_val), 5) if rail_val else 0.03546,
    },
    "car_occupancy_dft_nts_2023": 1.55,
    "car_average_per_pax_km": round(float(car_avg_val) / 1.55, 5),
    "notes": [
        "Bus, rail, coach: kg CO2e per passenger-km",
        "Car: kg CO2e per vehicle-km (divide by 1.55 occupancy for per-pax-km)",
        "Modal shift CO2 benefit = (car_per_pax_km - bus_per_pax_km) * distance_km"
    ]
}

In [13]:
out_path = DATA / "audit/desnz_carbon_constants.json"
with open(out_path, 'w') as f:
    json.dump(constants, f, indent=2)
print(f"Saved: {out_path}")
print(json.dumps(constants, indent=2))
print("03g_desnz_carbon COMPLETE")

Saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/desnz_carbon_constants.json
{
  "source": "DESNZ GHG Conversion Factors 2025 (condensed set)",
  "file": "ghg-conversion-factors-2025-condensed-set.xlsx",
  "published": "June 2025",
  "emission_factors_kg_co2e": {
    "bus_average_local_per_pax_km": 0.10385,
    "bus_london_per_pax_km": 0.06875,
    "coach_per_pax_km": 0.02776,
    "car_average_per_km": 0.17304,
    "national_rail_per_pax_km": 0.03546
  },
  "car_occupancy_dft_nts_2023": 1.55,
  "car_average_per_pax_km": 0.11164,
  "notes": [
    "Bus, rail, coach: kg CO2e per passenger-km",
    "Car: kg CO2e per vehicle-km (divide by 1.55 occupancy for per-pax-km)",
    "Modal shift CO2 benefit = (car_per_pax_km - bus_per_pax_km) * distance_km"
  ]
}
03g_desnz_carbon COMPLETE
